# Deepfake Detection Project

## Dataset Audit

This notebook verifies the structure, labels, image counts, source-video information, identities, and possible data leakage in the preprocessed Celeb-DF V2 image dataset before model training.


In [1]:
from pathlib import Path
from collections import Counter
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = Path("../data/Celeb_V2")
SEED = 42

In [2]:
# Print folder structure only

for root, dirs, files in os.walk(DATA_ROOT):
    # Ignore Jupyter checkpoint folders
    dirs[:] = [d for d in dirs if d != '.ipynb_checkpoints']

    level = root.replace(str(DATA_ROOT), '').count(os.sep)
    indent = '    ' * level

    print(f"{indent}{os.path.basename(root)}/")

Celeb_V2/
    Test/
        fake/
        real/
    Train/
        fake/
        real/
    Val/
        fake/
        real/


In [3]:
folders = [
    "Train/real",
    "Train/fake",
    "Val/real",
    "Val/fake",
    "Test/real",
    "Test/fake"
]

for folder in folders:
    path = DATA_ROOT / folder
    count = len(list(path.glob("*.jpg"))) if path.exists() else 0
    print(f"{folder}: {count}")
    

Train/real: 40288
Train/fake: 40536
Val/real: 5036
Val/fake: 5068
Test/real: 5036
Test/fake: 5067


In [4]:
# Check for non-.jpg files
print(Counter(p.suffix.lower() for p in DATA_ROOT.rglob("*")
              if p.is_file() and ".ipynb_checkpoints" not in p.parts))

Counter({'.jpg': 101031})


In [5]:
folders = {
    "Train/real": ("train", "real"),
    "Train/fake": ("train", "fake"),
    "Val/real": ("val", "real"),
    "Val/fake": ("val", "fake"),
    "Test/real": ("test", "real"),
    "Test/fake": ("test", "fake")
}

In [10]:
FAKE_RE = re.compile(r"^(id\d+)_(id\d+)_(\d{4})_face_\d+$")    # Celeb-synthesis: id{target}_id{source}_{clip}
REAL_RE = re.compile(r"^(?:(id\d+)_)?(\d{4,5})_face_\d+$")      # Celeb-real: id{n}_{clip} | YouTube-real: {video}

def parse_filename(filename, label):
    stem = Path(filename).stem
    if label == "fake":
        m = FAKE_RE.match(stem)
        if not m:
            return None, None, None
        return f"{m[1]}_{m[2]}_{m[3]}", m[1], m[2]       # video_id, target_id, source_id
    m = REAL_RE.match(stem)
    if not m:
        return None, None, None
    if m[1]:                                              # Celeb-real: identity known
        return f"{m[1]}_{m[2]}", m[1], None
    return m[2], None, None                               # YouTube-real: no identity

In [11]:
rows = []
for folder, (split, label) in folders.items():
    for image_path in (DATA_ROOT / folder).glob("*.jpg"):
        video_id, target_id, source_id = parse_filename(image_path.name, label)
        rows.append({
            "path": image_path.relative_to(DATA_ROOT).as_posix(),   # portable path
            "filename": image_path.name,
            "label": label,
            "split": split,
            "video_id": video_id,
            "target_id": target_id,
            "source_id": source_id,
        })
manifest["real_source"] = np.select(
    [(manifest.label == "real") & manifest.target_id.notna(), manifest.label == "real"],
    ["celeb_real", "youtube_real"], default="fake")
manifest = pd.DataFrame(rows)
manifest.head()

,path,filename,label,split,video_id,target_id,source_id
0,Train/real/00000_face_0.jpg,00000_face_0.jpg,real,train,00000,NaN,NaN
1,Train/real/00000_face_1.jpg,00000_face_1.jpg,real,train,00000,NaN,NaN
2,Train/real/00000_face_10.jpg,00000_face_10.jpg,real,train,00000,NaN,NaN
3,Train/real/00000_face_11.jpg,00000_face_11.jpg,real,train,00000,NaN,NaN
4,Train/real/00000_face_12.jpg,00000_face_12.jpg,real,train,00000,NaN,NaN


In [12]:
print("Total images:", len(manifest))

print("\nImages by label:")
print(manifest["label"].value_counts())

print("\nImages by split:")
print(manifest["split"].value_counts())

print("\nSplit + label counts:")
print(pd.crosstab(
    manifest["split"],
    manifest["label"]
))

Total images: 101031

Images by label:
label
fake    50671
real    50360
Name: count, dtype: int64

Images by split:
split
train    80824
val      10104
test     10103
Name: count, dtype: int64

Split + label counts:
label   fake   real
split              
test    5067   5036
train  40536  40288
val     5068   5036


In [13]:
# Every file name must be parsed
unparsed = manifest[manifest["video_id"].isna()]
print("Unparsed file names:", len(unparsed))
assert len(unparsed) == 0, unparsed["filename"].head(20).tolist()

Unparsed file names: 0


In [14]:
# Integrity + size check on ALL images (takes a few minutes)
bad, sizes = [], Counter()
for p in manifest["path"]:
    try:
        with Image.open(DATA_ROOT / p) as im:
            sizes[im.size] += 1
            im.verify()
    except Exception:
        bad.append(p)
print("Corrupt:", len(bad), "| Sizes:", sizes.most_common(3))
manifest = manifest[~manifest["path"].isin(bad)].reset_index(drop=True)

Corrupt: 0 | Sizes: [((256, 256), 101031)]


In [27]:
video_split_count = (
    manifest
    .groupby(["label", "video_id"])["split"]
    .nunique()
)

video_overlap = video_split_count[
    video_split_count > 1
]

print(
    "Number of source video IDs appearing "
    "in more than one split:",
    len(video_overlap)
)

Number of source video IDs appearing in more than one split: 5741


In [28]:
video_overlap.head(20)

label  video_id     
fake   id0_id16_0000    2
       id0_id16_0001    3
       id0_id16_0002    2
       id0_id16_0003    2
       id0_id16_0004    3
       id0_id16_0005    2
       id0_id16_0006    2
       id0_id16_0007    2
       id0_id16_0008    3
       id0_id16_0009    2
       id0_id17_0000    2
       id0_id17_0001    3
       id0_id17_0002    2
       id0_id17_0003    2
       id0_id17_0005    2
       id0_id17_0006    3
       id0_id17_0009    2
       id0_id1_0000     2
       id0_id1_0001     2
       id0_id1_0002     2
Name: split, dtype: int64

In [29]:
overlap_video_ids = (
    video_overlap
    .reset_index()[["label", "video_id"]]
)

overlap_details = manifest.merge(
    overlap_video_ids,
    on=["label", "video_id"]
)

overlap_summary = (
    overlap_details
    .groupby(["label", "video_id"])["split"]
    .unique()
)

overlap_summary.head(20)

label  video_id     
fake   id0_id16_0000         [train, test]
       id0_id16_0001    [train, val, test]
       id0_id16_0002         [train, test]
       id0_id16_0003          [train, val]
       id0_id16_0004    [train, val, test]
       id0_id16_0005          [train, val]
       id0_id16_0006          [train, val]
       id0_id16_0007          [train, val]
       id0_id16_0008    [train, val, test]
       id0_id16_0009         [train, test]
       id0_id17_0000          [train, val]
       id0_id17_0001    [train, val, test]
       id0_id17_0002          [train, val]
       id0_id17_0003          [train, val]
       id0_id17_0005          [train, val]
       id0_id17_0006    [train, val, test]
       id0_id17_0009         [train, test]
       id0_id1_0000           [train, val]
       id0_id1_0001          [train, test]
       id0_id1_0002           [train, val]
Name: split, dtype: object

In [30]:
fake_manifest = manifest[
    manifest["identity"].notna()
].copy()

identity_split_count = (
    fake_manifest
    .groupby("identity")["split"]
    .nunique()
)

identity_overlap = identity_split_count[
    identity_split_count > 1
]

print(
    "Fake identity pairs appearing "
    "in more than one split:",
    len(identity_overlap)
)

Fake identity pairs appearing in more than one split: 682


In [32]:
manifest["original_split"] = manifest["split"]

manifest.head()

,path,filename,label,split,video_id,identity,original_split
0,..\data\Celeb_V2\Train\real\00000_face_0.jpg,00000_face_0.jpg,real,train,00000,NaN,train
1,..\data\Celeb_V2\Train\real\00000_face_1.jpg,00000_face_1.jpg,real,train,00000,NaN,train
2,..\data\Celeb_V2\Train\real\00000_face_10.jpg,00000_face_10.jpg,real,train,00000,NaN,train
3,..\data\Celeb_V2\Train\real\00000_face_11.jpg,00000_face_11.jpg,real,train,00000,NaN,train
4,..\data\Celeb_V2\Train\real\00000_face_12.jpg,00000_face_12.jpg,real,train,00000,NaN,train


In [36]:
# Create group_key efficiently

manifest["group_key"] = np.where(
    manifest["label"] == "fake",
    "fake_" + manifest["identity"].astype(str),
    "real_" + manifest["video_id"].astype(str)
)

manifest[
    ["filename", "label", "video_id", "identity", "group_key"]
].head(10)

,filename,label,video_id,identity,group_key
0,00000_face_0.jpg,real,00000,NaN,real_00000
1,00000_face_1.jpg,real,00000,NaN,real_00000
2,00000_face_10.jpg,real,00000,NaN,real_00000
3,00000_face_11.jpg,real,00000,NaN,real_00000
4,00000_face_12.jpg,real,00000,NaN,real_00000
5,00000_face_13.jpg,real,00000,NaN,real_00000
6,00000_face_15.jpg,real,00000,NaN,real_00000
7,00000_face_16.jpg,real,00000,NaN,real_00000
8,00000_face_17.jpg,real,00000,NaN,real_00000
9,00000_face_18.jpg,real,00000,NaN,real_00000


In [37]:
manifest[
    manifest["label"] == "fake"
][
    ["filename", "label", "video_id", "identity", "group_key"]
].head(10)

,filename,label,video_id,identity,group_key
40288,id0_id16_0000_face_0.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40289,id0_id16_0000_face_1.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40290,id0_id16_0000_face_2.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40291,id0_id16_0000_face_3.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40292,id0_id16_0000_face_4.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40293,id0_id16_0000_face_5.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40294,id0_id16_0000_face_7.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40295,id0_id16_0000_face_8.jpg,fake,id0_id16_0000,id0_id16,fake_id0_id16
40296,id0_id16_0001_face_10.jpg,fake,id0_id16_0001,id0_id16,fake_id0_id16
40297,id0_id16_0001_face_11.jpg,fake,id0_id16_0001,id0_id16,fake_id0_id16


In [38]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

def grouped_split(data, group_column, random_state=42):
    # 80% train, 20% temporary
    splitter1 = GroupShuffleSplit(
        n_splits=1,
        train_size=0.80,
        random_state=random_state
    )

    train_idx, temp_idx = next(
        splitter1.split(
            data,
            groups=data[group_column]
        )
    )

    train_data = data.iloc[train_idx].copy()
    temp_data = data.iloc[temp_idx].copy()

    # Split remaining 20% equally into validation and test
    splitter2 = GroupShuffleSplit(
        n_splits=1,
        train_size=0.50,
        random_state=random_state
    )

    val_idx, test_idx = next(
        splitter2.split(
            temp_data,
            groups=temp_data[group_column]
        )
    )

    val_data = temp_data.iloc[val_idx].copy()
    test_data = temp_data.iloc[test_idx].copy()

    train_data["clean_split"] = "train"
    val_data["clean_split"] = "val"
    test_data["clean_split"] = "test"

    return pd.concat(
        [train_data, val_data, test_data],
        ignore_index=True
    )

In [39]:
real_data = manifest[
    manifest["label"] == "real"
].copy()

fake_data = manifest[
    manifest["label"] == "fake"
].copy()

real_clean = grouped_split(
    real_data,
    group_column="group_key",
    random_state=42
)

fake_clean = grouped_split(
    fake_data,
    group_column="group_key",
    random_state=42
)

clean_manifest = pd.concat(
    [real_clean, fake_clean],
    ignore_index=True
)

In [40]:
print(pd.crosstab(
    clean_manifest["clean_split"],
    clean_manifest["label"]
))

label         fake   real
clean_split              
test          5179   5020
train        40619  40375
val           4873   4965


In [41]:
video_check = (
    clean_manifest
    .groupby(["label", "video_id"])["clean_split"]
    .nunique()
)

video_leakage = video_check[
    video_check > 1
]

print(
    "Video IDs appearing in more than one CLEAN split:",
    len(video_leakage)
)

Video IDs appearing in more than one CLEAN split: 0


In [42]:
fake_clean_manifest = clean_manifest[
    clean_manifest["label"] == "fake"
]

identity_check = (
    fake_clean_manifest
    .groupby("identity")["clean_split"]
    .nunique()
)

identity_leakage = identity_check[
    identity_check > 1
]

print(
    "Fake identity pairs appearing in more than one CLEAN split:",
    len(identity_leakage)
)

Fake identity pairs appearing in more than one CLEAN split: 0


In [44]:
clean_manifest.to_csv(
    "../data/celeb_df_clean_manifest.csv",
    index=False
)

print("Clean manifest saved.")

Clean manifest saved.


In [45]:
split_summary = pd.crosstab(
    clean_manifest["clean_split"],
    clean_manifest["label"]
)

print(split_summary)

label         fake   real
clean_split              
test          5179   5020
train        40619  40375
val           4873   4965


In [46]:
split_percent = (
    clean_manifest["clean_split"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print(split_percent)

clean_split
test     10.094921
train    80.167473
val       9.737605
Name: proportion, dtype: float64


### Dataset Leakage Audit Summary

The original preprocessed Celeb-DF V2 train, validation, and test folders contained substantial overlap at the source-video level. A total of 5,741 source video IDs appeared in more than one split, and 682 fake identity pairs were also distributed across multiple splits.

To reduce data leakage, a new grouped train/validation/test split was created. For real images, all frames originating from the same source video were assigned to a single partition. For fake images, all frames associated with the same fake identity pair were assigned to a single partition.

The cleaned dataset uses approximately an 80/10/10 train, validation, and test split. After rebuilding the partitions, the number of source video IDs appearing in more than one clean split was 0, and the number of fake identity pairs appearing in more than one clean split was also 0.

A limitation is that the real-image filenames do not expose celebrity identity information, so the real partition is video-disjoint but cannot be guaranteed to be identity-disjoint.

In [1]:
gitignore_content = """data/Celeb_V2/
models/*.keras
models/*.h5
.ipynb_checkpoints/
**/.ipynb_checkpoints/
__pycache__/
*.pyc
Thumbs.db
.DS_Store
"""

with open("../.gitignore", "w") as f:
    f.write(gitignore_content)

print(".gitignore created successfully")

.gitignore created successfully


In [2]:
from pathlib import Path

gitignore_path = Path("../.gitignore")

print("Exists:", gitignore_path.exists())
print("Full path:", gitignore_path.resolve())

Exists: True
Full path: C:\Users\urshi\Downloads\PDS\Project\.gitignore
